In [20]:
%load_ext autoreload
%autoreload 2

In [23]:
import scraper
print(scraper.__file__)

d:\Desktop\Ai  Engineer Core Track\Sales Brochure Generator\scraper.py


In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
ollama = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama', 
)

# Use 'llama3.2' as the model name
MODEL = 'qwen2.5'


### for Open Ai Api 


In [4]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')



if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:

    print("API key looks good so far")

else:

    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

    

MODEL = 'gpt-5-nano'

openai = OpenAI()

API key looks good so far


In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model='gpt-5-nano',
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'external company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

## Second Step  make the brochure   


In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    
    # Use .get() to safely check for the key
    links = relevant_links.get('links', [])
    
    for link in links:
        result += f"\n\n### Link: {link.get('type', 'Unknown')}\n"
        result += fetch_website_contents(link.get("url", ""))
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://cricbuzz.com"))

## Landing Page:

IPL 2026 | Live Cricket Score, Schedule, News, Stats &amp; Videos  | Cricbuzz.com

Menu
Live Scores
Schedule
Archives
News
Series
Teams
Videos
Rankings
More
MATCHES
NZW
vs
WIW
-
Need 98 off 65b
AFG
vs
IND
-
IND won
AUSW
vs
RSAW
-
AUSW won
INDW
vs
PAKW
-
Preview
WI
vs
SL
-
Preview
ALL
All
Live Now
Today
INTERNATIONAL
Afghanistan tour of India 2026
Afghanistan vs India
1st ODI
Bangladesh v Australia, 2026
Bangladesh vs Australia
3rd ODI
West Indies v Sri Lanka 2026
West Indies vs Sri Lanka
2nd T20I
ICC CWC League Two 2023-27
Canada vs United States of America
113th Match
United States of America vs Netherlands
114th Match
Viking Cup
Finland vs Norway
5th Match
Sweden vs Austria
6th Match
Norway vs Austria
3rd place play-off
Finland vs Sweden
Final
LEAGUE
T20 Mumbai 2026
MSC Maratha Royals vs ARCS Andheri
Final
MPPL 2026
Jabalpur Royal Lions vs Malwa Stallions
23rd Match
Gwalior Cheetahs vs Bundelkhand Bulls
24th Match
Ujjain Falcons vs Rewa Jaguars
25th Match
Chambal Gh

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://airbnb.com")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nAirbnb: Holiday Rentals, Cabins, Beach Houses, Unique Homes & Experiences\n\nSkip to content\nWe’re sorry, some parts of the Airbnb website don’t work properly without JavaScript enabled.\nAirbnb homepage\nHomes\nHomes\nExperiences\nExperiences, new\nServices\nServices, new\nWhere\nWhen\nAdd dates\nWho\nAdd guests\nBecome a host\nHelp Centre\nBecome a host\nIt’s easy to start hosting and earn extra income.\nFind a co-host\nLog in or sign up\nHomes on Airbnb\n0 of 0 items showing\nHomes on Airbnb\n0 of 0 items showing\nHomes on Airbnb\n0 of 0 items showing\n## Relevant Links:\n\n\n### Link: homepage\nAirbnb: Holiday Rentals, Cabins, Beach Houses, Unique Homes & Experiences\n\nSkip to content\nWe’re sorry, some parts of the Airbnb website don’t work 

###  Creat the brochure


In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model='gpt-5-nano',
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("Make my trip", "https://makemytrip.com")

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model='gpt-5-nano',
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Make my Trip", "https://makemytrip.com")